# Topic 2 Temperature Prediction MVP

## Part 1: Project plan

- Goal: predict next-season temperature anomaly from current-season climate indices.
- Target data: ERA5 monthly 2m temperature.
- Predictors: Nino 3.4, PDO, AO.
- Method for MVP: simple multiple linear regression.
- This notebook is only a first MVP, not the final analysis.

## Part 2: Load packages

In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

## Part 3: Load data

In [ ]:
data_dir = "data"

era5_path = f"{data_dir}/ERA5_2mtemp_1x1.nc"
nino34_path = f"{data_dir}/nina34.anom.data"
pdo_path = f"{data_dir}/ersst.v5.pdo.dat"
ao_path = f"{data_dir}/ao.long.csv"

In [ ]:
def read_monthly_index_file(path, value_name):
    rows = []
    
    with open(path, "r") as file:
        for line in file:
            parts = line.split()
            if len(parts) < 13 or not parts[0].isdigit():
                continue
            
            year = int(parts[0])
            values = [float(value) for value in parts[1:13]]
            
            for month, value in enumerate(values, start=1):
                if value in [-99.99, 99.99]:
                    value = np.nan
                rows.append({"date": pd.Timestamp(year, month, 1), value_name: value})
    
    return pd.DataFrame(rows).set_index("date")[value_name].sort_index()

In [ ]:
era5_dataset = xr.open_dataset(era5_path)

nino34 = read_monthly_index_file(nino34_path, "nino34")
pdo = read_monthly_index_file(pdo_path, "pdo")

ao_data = pd.read_csv(ao_path)
ao_data.columns = ["date", "ao"]
ao_data["date"] = pd.to_datetime(ao_data["date"])
ao_data["ao"] = pd.to_numeric(ao_data["ao"], errors="coerce").replace(-999, np.nan)
ao = ao_data.set_index("date")["ao"].sort_index()

era5_dataset

## Part 4: Simple temperature target

In [ ]:
latitude = 40.0
longitude = -105.3

temperature_name = list(era5_dataset.data_vars)[0]
temperature = era5_dataset[temperature_name]

# ERA5 longitudes may use 0 to 360.
if float(temperature.lon.max()) > 180 and longitude < 0:
    longitude_for_data = longitude % 360
else:
    longitude_for_data = longitude

boulder_temperature = temperature.sel(
    lat=latitude,
    lon=longitude_for_data,
    method="nearest"
)

boulder_temperature_c = boulder_temperature - 273.15
monthly_temperature = boulder_temperature_c.to_series().dropna()
monthly_temperature.index = pd.to_datetime(monthly_temperature.index)

# Remove the simple monthly climatology.
monthly_temperature_anomaly = monthly_temperature - monthly_temperature.groupby(monthly_temperature.index.month).transform("mean")
monthly_temperature_anomaly.name = "temperature_anomaly"

print("Nearest latitude:", float(boulder_temperature.lat.values))
print("Nearest longitude:", float(boulder_temperature.lon.values))
monthly_temperature_anomaly.head()

## Part 5: Make seasonal data

In [ ]:
seasonal_temperature = monthly_temperature_anomaly.resample("QS-DEC").mean()
seasonal_nino34 = nino34.resample("QS-DEC").mean()
seasonal_pdo = pdo.resample("QS-DEC").mean()
seasonal_ao = ao.resample("QS-DEC").mean()

seasonal_data = pd.concat(
    [seasonal_nino34, seasonal_pdo, seasonal_ao, seasonal_temperature],
    axis=1
)

seasonal_data.head()

## Part 6: Make one-season-ahead target

In [ ]:
model_data = seasonal_data.copy()
model_data["temperature_anomaly_next_season"] = model_data["temperature_anomaly"].shift(-1)

model_data = model_data[
    ["nino34", "pdo", "ao", "temperature_anomaly_next_season"]
].dropna()

model_data.head()

## Part 7: Very simple EDA

In [ ]:
model_data.head()

In [ ]:
model_data.describe()

In [ ]:
model_data.corr()

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(model_data.index, model_data["temperature_anomaly_next_season"])
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Next-season temperature anomaly")
plt.xlabel("Season")
plt.ylabel("Temperature anomaly (C)")
plt.show()

In [ ]:
plt.figure(figsize=(5, 4))
plt.scatter(model_data["nino34"], model_data["temperature_anomaly_next_season"], alpha=0.7)
plt.axhline(0, color="black", linewidth=0.8)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Nino 3.4 vs next-season temperature")
plt.xlabel("Nino 3.4")
plt.ylabel("Next-season temperature anomaly (C)")
plt.show()

## Part 8: Simple regression model

In [ ]:
features = ["nino34", "pdo", "ao"]
target = "temperature_anomaly_next_season"

split_index = int(len(model_data) * 0.8)
train_data = model_data.iloc[:split_index]
test_data = model_data.iloc[split_index:]

X_train = train_data[features]
y_train = train_data[target]
X_test = test_data[features]
y_test = test_data[target]

model = LinearRegression()
model.fit(X_train, y_train)

test_predictions = model.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))
test_r2 = r2_score(y_test, test_predictions)

coefficients = pd.Series(model.coef_, index=features, name="coefficient")

print("Coefficients")
print(coefficients)
print("Intercept:", model.intercept_)
print("Test RMSE:", test_rmse)
print("Test R2:", test_r2)

## Part 9: Simple prediction plot

In [ ]:
prediction_data = test_data.copy()
prediction_data["predicted_temperature_anomaly_next_season"] = test_predictions

plt.figure(figsize=(10, 4))
plt.plot(
    prediction_data.index,
    prediction_data["temperature_anomaly_next_season"],
    label="Observed"
)
plt.plot(
    prediction_data.index,
    prediction_data["predicted_temperature_anomaly_next_season"],
    label="Predicted"
)
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Observed vs predicted next-season temperature anomaly")
plt.xlabel("Season")
plt.ylabel("Temperature anomaly (C)")
plt.legend()
plt.show()

## Part 10: Short notes for professor

- This is a first MVP.
- Next step: check index parsing carefully.
- Next step: add better out-of-sample validation.
- Next step: assess residuals.
- Next step: compare models with different predictors.
- Next step: maybe test different locations or seasons.